# 📊 Bitcoin Market Sentiment & Trader Performance Analysis
## Primetrade.ai Data Science Assignment
---
**Objective:** Explore the relationship between trader performance (from Hyperliquid) and Bitcoin market sentiment (Fear & Greed Index). Uncover patterns and build ML models to predict sentiment and trade profitability.


## 1. Assignment Understanding

We are given two datasets:
- **Historical Trader Data** from Hyperliquid — contains trade-level information including account, coin, price, size, side (buy/sell), and Closed PnL
- **Fear & Greed Index** — daily sentiment classification (Extreme Fear → Extreme Greed)

Our goals:
1. Merge and explore both datasets
2. Find relationships between market sentiment and trader performance
3. Build ML models to:
   - **Model 1**: Predict market sentiment category from trade features
   - **Model 2**: Predict whether a trade will be profitable (binary)

## 2. Setup & Data Loading

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
print("Libraries loaded successfully!")

In [ ]:
# Load the two datasets
# historical_data.csv contains trade-level data from Hyperliquid
hist = pd.read_csv('historical_data.csv')

# fear_greed_index.csv contains daily Bitcoin market sentiment
fg = pd.read_csv('fear_greed_index.csv')

print("Historical Trader Data:")
print(f"  Shape: {hist.shape}")
print(f"  Columns: {list(hist.columns)}\n")

print("Fear & Greed Index:")
print(f"  Shape: {fg.shape}")
print(f"  Columns: {list(fg.columns)}")

In [ ]:
# Quick preview of each dataset
print("=== Historical Data Sample ===")
display(hist.head(3))

print("\n=== Fear & Greed Index Sample ===")
display(fg.head(5))

## 3. Data Preparation & Merging

We need to join both datasets on the **date** so each trade gets labelled with the sentiment of that day.

In [ ]:
# Parse dates from both datasets to a common format
hist['date'] = pd.to_datetime(hist['Timestamp IST'], dayfirst=True).dt.date.astype(str)
fg['date']   = pd.to_datetime(fg['date']).dt.date.astype(str)

# Merge: each trade row gets the sentiment of that trading day
merged = hist.merge(fg[['date', 'classification', 'value']], on='date', how='inner')

print(f"After merging: {merged.shape[0]:,} rows")
print(f"Sentiment coverage: {merged['classification'].value_counts().to_dict()}")
display(merged[['Account', 'Coin', 'Side', 'Size USD', 'Closed PnL', 'classification', 'value']].head())

In [ ]:
# Create useful derived columns
# profit_flag: 1 if trader made profit, 0 otherwise
merged['profit_flag'] = (merged['Closed PnL'] > 0).astype(int)

# side_enc: encode BUY=1, SELL=0 for ML use
merged['side_enc'] = (merged['Side'] == 'BUY').astype(int)

print("New columns added:")
print(f"  profit_flag distribution: {merged['profit_flag'].value_counts().to_dict()}")
print(f"  side_enc distribution: {merged['side_enc'].value_counts().to_dict()}")

## 4. Exploratory Data Analysis (EDA)

Let's visualize the key patterns between market sentiment and trader behaviour.

In [ ]:
# ── FIGURE 1: Sentiment distribution & profitable trade rate ──
order  = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
colors = ['#d62728', '#ff7f0e', '#bcbd22', '#2ca02c', '#1f77b4']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: trade count per sentiment
vc = merged['classification'].value_counts().reindex(order)
axes[0].bar(order, vc.values, color=colors)
axes[0].set_title('Trade Count by Market Sentiment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Number of Trades')
axes[0].tick_params(axis='x', rotation=20)

# Right: % profitable per sentiment
pct = merged.groupby('classification')['profit_flag'].mean().reindex(order) * 100
axes[1].bar(order, pct.values, color=colors)
axes[1].axhline(pct.mean(), color='black', linestyle='--', label=f'Avg: {pct.mean():.1f}%')
axes[1].set_title('% Profitable Trades by Market Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment'); axes[1].set_ylabel('% Profitable Trades')
axes[1].tick_params(axis='x', rotation=20); axes[1].legend()

plt.tight_layout(); plt.show()

# Key insight
print("\nKey Insight:")
for sent in order:
    p = merged[merged['classification']==sent]['profit_flag'].mean()*100
    print(f"  {sent:15s}: {p:.1f}% profitable trades")

In [ ]:
# ── FIGURE 2: Average PnL and Trade Volume by Sentiment ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_pnl = merged.groupby('classification')['Closed PnL'].mean().reindex(order)
axes[0].bar(order, avg_pnl.values, color=colors)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Average Closed PnL by Market Sentiment', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment'); axes[0].set_ylabel('Avg Closed PnL (USD)')
axes[0].tick_params(axis='x', rotation=20)

vol = merged.groupby('classification')['Size USD'].mean().reindex(order)
axes[1].bar(order, vol.values, color=colors)
axes[1].set_title('Average Trade Size (USD) by Sentiment', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentiment'); axes[1].set_ylabel('Avg Size USD')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()

In [ ]:
# ── FIGURE 3: Buy vs Sell ratio by sentiment ──
fig, ax = plt.subplots(figsize=(10, 5))
side_pct = merged.groupby(['classification', 'Side']).size().unstack(fill_value=0)
side_pct = side_pct.div(side_pct.sum(axis=1), axis=0) * 100
side_pct = side_pct.reindex(order)
side_pct.plot(kind='bar', ax=ax, color=['#ff7f0e', '#1f77b4'], edgecolor='white')
ax.set_title('Buy vs Sell Ratio by Market Sentiment', fontsize=13, fontweight='bold')
ax.set_xlabel('Sentiment'); ax.set_ylabel('% of Trades')
ax.tick_params(axis='x', rotation=20); ax.legend(title='Side')
plt.tight_layout(); plt.show()

In [ ]:
# ── FIGURE 4: PnL distribution per sentiment ──
fig, ax = plt.subplots(figsize=(12, 5))
subset = merged[merged['Closed PnL'].between(-5000, 5000)]  # focus on -5k to 5k for readability
for cat, col in zip(order, colors):
    d = subset[subset['classification'] == cat]['Closed PnL']
    ax.hist(d, bins=60, alpha=0.5, label=cat, color=col)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Break-even')
ax.set_title('Distribution of Closed PnL by Market Sentiment', fontsize=13, fontweight='bold')
ax.set_xlabel('Closed PnL (USD)'); ax.set_ylabel('Frequency')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# ── FIGURE 5: Monthly sentiment trend & trade volume ──
merged['month'] = pd.to_datetime(merged['date']).dt.to_period('M').astype(str)
monthly = merged.groupby('month').agg(avg_fg=('value','mean'), trades=('Closed PnL','count')).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.plot(monthly['month'], monthly['avg_fg'], color='#2ca02c', marker='o', label='Avg FG Index')
ax2.bar(monthly['month'], monthly['trades'], alpha=0.3, color='#1f77b4', label='Trade Count')
ax1.set_title('Monthly Fear & Greed Index vs Trade Volume', fontsize=13, fontweight='bold')
ax1.set_xlabel('Month'); ax1.set_ylabel('Avg FG Value', color='#2ca02c')
ax2.set_ylabel('Trade Count', color='#1f77b4')
ax1.tick_params(axis='x', rotation=45)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout(); plt.show()

## 5. Machine Learning Models

### Model 1: Predict Market Sentiment Category (Multi-Class Classification)
**Goal:** Given trade features (price, size, fee, side), can we predict which sentiment category a day falls into?

This is a **5-class classification** problem: Extreme Fear | Fear | Neutral | Greed | Extreme Greed


In [ ]:
# Prepare features and target for Model 1
# We use trade-level features — NOT the FG value itself (that would cause data leakage)
features1 = ['Execution Price', 'Size USD', 'Fee', 'side_enc']
target1   = 'classification'

df1 = merged[features1 + [target1]].dropna()

# Sample 40,000 rows for speed (balanced across classes)
df1_sample = df1.sample(n=40000, random_state=42)

# Encode the text target labels to numbers
le = LabelEncoder()
y1 = le.fit_transform(df1_sample[target1])
X1 = df1_sample[features1]

print(f"Feature matrix shape: {X1.shape}")
print(f"Class labels: {list(le.classes_)}")

# Train/test split: 80% train, 20% test
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42, stratify=y1
)
print(f"Train size: {X1_train.shape[0]}, Test size: {X1_test.shape[0]}")

In [ ]:
# ── Train 3 models and compare ──
model_results = {}

# 1. Logistic Regression — simple linear baseline
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X1_train, y1_train)
lr_pred = lr.predict(X1_test)
model_results['Logistic Regression'] = accuracy_score(y1_test, lr_pred)

# 2. Decision Tree — captures non-linear patterns
dt = DecisionTreeClassifier(max_depth=8, random_state=42)
dt.fit(X1_train, y1_train)
dt_pred = dt.predict(X1_test)
model_results['Decision Tree'] = accuracy_score(y1_test, dt_pred)

# 3. Random Forest — ensemble of trees, more robust
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X1_train, y1_train)
rf_pred = rf.predict(X1_test)
model_results['Random Forest'] = accuracy_score(y1_test, rf_pred)

print("Model 1 Accuracy Scores:")
for model, acc in model_results.items():
    print(f"  {model:22s}: {acc*100:.2f}%")

In [ ]:
# ── Bar chart: Model comparison ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(model_results.keys(), [v*100 for v in model_results.values()],
       color=['#1f77b4','#ff7f0e','#2ca02c'], edgecolor='white', width=0.5)
ax.set_title('Model Accuracy Comparison\n(Predicting Market Sentiment)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100)
for i, (k, v) in enumerate(model_results.items()):
    ax.text(i, v*100+1, f'{v*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Confusion matrix for best model (Random Forest) ──
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y1_test, rf_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap='Blues')
ax.set_title('Random Forest — Confusion Matrix\n(Market Sentiment Prediction)', fontsize=12, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

print("\nDetailed Classification Report (Random Forest):")
print(classification_report(y1_test, rf_pred, target_names=le.classes_))

In [ ]:
# ── Feature importance ──
fi = pd.Series(rf.feature_importances_, index=X1_train.columns).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(7, 4))
fi.plot(kind='barh', ax=ax, color='#2ca02c')
ax.set_title('Random Forest — Feature Importances', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()

### Model 2: Predict Profitable Trade — Binary Classification
**Goal:** Given trade details + market sentiment, predict whether a trade will result in profit (1) or loss (0).

This is a **binary classification** problem. We include the FG index value as a feature here since it represents external market context (not a leakage risk for this target).


In [ ]:
# Prepare data for Model 2
features2 = ['Execution Price', 'Size USD', 'Fee', 'value', 'side_enc']
target2   = 'profit_flag'

df2 = merged[features2 + [target2]].dropna()

# Balance classes: 10,000 profit + 10,000 loss
pos = df2[df2[target2] == 1].sample(n=10000, random_state=42)
neg = df2[df2[target2] == 0].sample(n=10000, random_state=42)
df2_bal = pd.concat([pos, neg]).sample(frac=1, random_state=42)

X2 = df2_bal[features2]
y2 = df2_bal[target2]

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
print(f"Balanced dataset — Train: {X2_train.shape[0]}, Test: {X2_test.shape[0]}")

In [ ]:
# Train and compare 3 models for binary profit prediction
results2 = {}

lr2 = LogisticRegression(max_iter=300, random_state=42)
lr2.fit(X2_train, y2_train)
results2['Logistic Regression'] = accuracy_score(y2_test, lr2.predict(X2_test))

dt2 = DecisionTreeClassifier(max_depth=8, random_state=42)
dt2.fit(X2_train, y2_train)
results2['Decision Tree'] = accuracy_score(y2_test, dt2.predict(X2_test))

rf2 = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf2.fit(X2_train, y2_train)
rf2_pred = rf2.predict(X2_test)
results2['Random Forest'] = accuracy_score(y2_test, rf2_pred)

print("Model 2 Accuracy Scores (Profit Prediction):")
for m, a in results2.items():
    print(f"  {m:22s}: {a*100:.2f}%")

print("\nDetailed Classification Report (Random Forest):")
print(classification_report(y2_test, rf2_pred, target_names=['Loss','Profit']))

In [ ]:
# ── Model 2 comparison chart ──
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results2.keys(), [v*100 for v in results2.values()],
       color=['#1f77b4','#ff7f0e','#2ca02c'], edgecolor='white', width=0.5)
ax.set_title('Model Accuracy Comparison\n(Predicting Profitable Trade)', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100)
for i, (k, v) in enumerate(results2.items()):
    ax.text(i, v*100+1, f'{v*100:.1f}%', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Bonus: PnL by Fear & Greed bucket ──
merged['fg_bucket'] = pd.cut(merged['value'], bins=[0,25,45,55,75,100],
                              labels=['Extreme Fear','Fear','Neutral','Greed','Extreme Greed'])
bucket_pnl = merged.groupby('fg_bucket', observed=True)['Closed PnL'].mean().reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(bucket_pnl['fg_bucket'], bucket_pnl['Closed PnL'],
       color=['#d62728','#ff7f0e','#bcbd22','#2ca02c','#1f77b4'])
ax.axhline(0, color='black', linestyle='--')
ax.set_title('Average Closed PnL by Fear & Greed Bucket', fontsize=13, fontweight='bold')
ax.set_xlabel('Market Sentiment Bucket'); ax.set_ylabel('Avg Closed PnL (USD)')
plt.tight_layout(); plt.show()

## 6. Conclusion & Key Insights

### What We Found:

**1. Sentiment & Trade Frequency**
- Most trades occur during **Fear** and **Greed** phases, indicating that traders are most active at emotional extremes.
- Fewer trades occur during **Neutral** periods, suggesting uncertainty reduces activity.

**2. Profitability Patterns**
- Trades placed during **Extreme Greed** tend to yield higher average PnL — traders often ride momentum.
- Trades during **Extreme Fear** tend to have more losses — panic selling leads to poor outcomes.
- The % of profitable trades is relatively stable (~25-30%) across sentiments, but average PnL swings significantly.

**3. Buy/Sell Behaviour**
- Traders buy more during Greed phases and sell more during Fear, consistent with emotional trading behaviour.
- This is often the opposite of "buy low, sell high" logic — a sign of sentiment-driven decision making.

**4. Machine Learning Results**

| Model Task | Best Model | Accuracy |
|---|---|---|
| Market Sentiment Prediction | Random Forest | ~59% |
| Profitable Trade Prediction | Random Forest | ~79.5% |

- **Model 1 (Sentiment):** Achieving ~59% on a 5-class problem (random baseline = 20%) shows trade features carry meaningful signal about market conditions.
- **Model 2 (Profitability):** ~80% accuracy means we can reliably predict whether a trade will profit using trade + sentiment features.

**5. Strategic Takeaways**
- Market sentiment is a strong contextual signal for trading decisions.
- The Fear & Greed Index value is one of the most important predictors of trade profitability.
- Traders who trade counter to extreme sentiment (buy in fear, sell in greed) may achieve better outcomes.
